In [6]:
from google.colab import drive
drive.mount('/content/drive')

ModuleNotFoundError: No module named 'google.colab'

In [ ]:
# %load nn.py
"""
The main code for the recurrent and convolutional networks assignment.

See README.md for details.
"""

from typing import Tuple, List, Dict

import tensorflow

def create_toy_rnn(input_shape: tuple, n_outputs: int) \
        -> Tuple[tensorflow.keras.models.Model, Dict]:
    """Creates a recurrent neural network for a toy problem.

    The network will take as input a sequence of number pairs, (x_{t}, y_{t}),
    where t is the time step. It must learn to produce x_{t-3} - y{t} as the
    output of time step t.

    This method does not call Model.fit, but the dictionary it returns alongside
    the model will be passed as extra arguments whenever Model.fit is called.
    This can be used to, for example, set the batch size or use early stopping.

    :param input_shape: The shape of the inputs to the model.
    :param n_outputs: The number of outputs from the model.
    :return: A tuple of (neural network, Model.fit keyword arguments)
    """
    model = tensorflow.keras.models.Sequential([
        tensorflow.keras.layers.SimpleRNN(64, input_shape=input_shape, return_sequences=True),
        tensorflow.keras.layers.LSTM(32, return_sequences=True),
        tensorflow.keras.layers.Dense(16, activation='relu'),
        tensorflow.keras.layers.Dense(n_outputs)
    ])

    model.compile(optimizer=tensorflow.keras.optimizers.Adam(learning_rate=0.01),
                  loss='mse',
                  metrics=['mae'])

    fit_kwargs = {
        'batch_size': 32,
        'epochs': 100,
        'validation_split': 0.2,
        'callbacks': [
            tensorflow.keras.callbacks.EarlyStopping(patience=10, restore_best_weights=True),
        ]
    }

    return model, fit_kwargs

def create_mnist_cnn(input_shape: tuple, n_outputs: int) \
        -> Tuple[tensorflow.keras.models.Model, Dict]:
    """Creates a convolutional neural network for digit classification.

    The network will take as input a 28x28 grayscale image, and produce as
    output one of the digits 0 through 9. The network will be trained and tested
    on a fraction of the MNIST data: http://yann.lecun.com/exdb/mnist/

    This method does not call Model.fit, but the dictionary it returns alongside
    the model will be passed as extra arguments whenever Model.fit is called.
    This can be used to, for example, set the batch size or use early stopping.

    :param input_shape: The shape of the inputs to the model.
    :param n_outputs: The number of outputs from the model.
    :return: A tuple of (neural network, Model.fit keyword arguments)
    """
    # Check for GPU availability
    if len(tensorflow.config.list_physical_devices('GPU')) > 0:
        # GPU is available, use float32
        tensorflow.keras.backend.set_floatx('float32')
    else:
        # No GPU, use float64
        tensorflow.keras.backend.set_floatx('float64')

    model = tensorflow.keras.models.Sequential([
        tensorflow.keras.layers.Conv2D(32, (3, 3), activation='relu', input_shape=input_shape),
        tensorflow.keras.layers.MaxPool2D((2, 2)),
        tensorflow.keras.layers.Conv2D(64, (3, 3), activation='relu'),
        tensorflow.keras.layers.MaxPool2D((2, 2)),
        tensorflow.keras.layers.Flatten(),
        tensorflow.keras.layers.Dense(64, activation='relu'),
        tensorflow.keras.layers.Dense(n_outputs, activation='softmax')
    ])

    model.compile(optimizer=tensorflow.keras.optimizers.Adam(learning_rate=0.001),
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])

    fit_kwargs = {
        'batch_size': 32,
        'epochs': 10,
        'validation_split': 0.2,
        'callbacks': [
            tensorflow.keras.callbacks.EarlyStopping(patience=3, restore_best_weights=True),
        ]
    }

    return model, fit_kwargs


def create_youtube_comment_rnn(vocabulary: List[str], n_outputs: int) \
        -> Tuple[tensorflow.keras.models.Model, Dict]:
    """Creates a recurrent neural network for spam classification.

    This network will take as input a YouTube comment, and produce as output
    either 1, for spam, or 0, for ham (non-spam). The network will be trained
    and tested on data from:
    https://archive.ics.uci.edu/ml/datasets/YouTube+Spam+Collection

    Each comment is represented as a series of tokens, with each token
    represented by a number, which is its index in the vocabulary. Note that
    comments may be of variable length, so in the input matrix, comments with
    fewer tokens than the matrix width will be right-padded with zeros.

    This method does not call Model.fit, but the dictionary it returns alongside
    the model will be passed as extra arguments whenever Model.fit is called.
    This can be used to, for example, set the batch size or use early stopping.

    :param vocabulary: The vocabulary defining token indexes.
    :param n_outputs: The number of outputs from the model.
    :return: A tuple of (neural network, Model.fit keyword arguments)
    """
    vocab_size = len(vocabulary)

    model = tensorflow.keras.models.Sequential([
        tensorflow.keras.layers.Embedding(vocab_size, 100, input_length=None),
        tensorflow.keras.layers.Bidirectional(tensorflow.keras.layers.LSTM(64, return_sequences=True)),
        tensorflow.keras.layers.Bidirectional(tensorflow.keras.layers.LSTM(32)),
        tensorflow.keras.layers.Dense(16, activation='relu'),
        tensorflow.keras.layers.Dense(n_outputs, activation='sigmoid')
    ])

    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

    fit_kwargs = {
        'batch_size': 32,
        'epochs': 10,
        'validation_split': 0.2,
        'callbacks': [tensorflow.keras.callbacks.EarlyStopping(patience=2)]
    }

    return model, fit_kwargs

def create_youtube_comment_cnn(vocabulary: List[str], n_outputs: int) \
        -> Tuple[tensorflow.keras.models.Model, Dict]:
    """Creates a convolutional neural network for spam classification.

    This network will take as input a YouTube comment, and produce as output
    either 1, for spam, or 0, for ham (non-spam). The network will be trained
    and tested on data from:
    https://archive.ics.uci.edu/ml/datasets/YouTube+Spam+Collection

    Each comment is represented as a series of tokens, with each token
    represented by a number, which is its index in the vocabulary. Note that
    comments may be of variable length, so in the input matrix, comments with
    fewer tokens than the matrix width will be right-padded with zeros.

    This method does not call Model.fit, but the dictionary it returns alongside
    the model will be passed as extra arguments whenever Model.fit is called.
    This can be used to, for example, set the batch size or use early stopping.

    :param vocabulary: The vocabulary defining token indexes.
    :param n_outputs: The number of outputs from the model.
    :return: A tuple of (neural network, Model.fit keyword arguments)
    """
    vocab_size = len(vocabulary)

    model = tensorflow.keras.models.Sequential([
        tensorflow.keras.layers.Embedding(vocab_size, 100, input_length=None),
        tensorflow.keras.layers.Conv1D(128, 5, activation='relu'),
        tensorflow.keras.layers.GlobalMaxPool1D(),
        tensorflow.keras.layers.Dense(64, activation='relu'),
        tensorflow.keras.layers.Dense(n_outputs, activation='sigmoid')
    ])

    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

    fit_kwargs = {
        'batch_size': 64,
        'epochs': 10,
        'validation_split': 0.2,
        'callbacks': [tensorflow.keras.callbacks.EarlyStopping(patience=2)]
    }

    return model, fit_kwargs


In [8]:
with open('nn.py', 'r') as file:
    code = file.read()

In [9]:
print(code)

"""
The main code for the recurrent and convolutional networks assignment.

See README.md for details.
"""

from typing import Tuple, List, Dict

import tensorflow

def create_toy_rnn(input_shape: tuple, n_outputs: int) \
        -> Tuple[tensorflow.keras.models.Model, Dict]:
    """Creates a recurrent neural network for a toy problem.

    The network will take as input a sequence of number pairs, (x_{t}, y_{t}),
    where t is the time step. It must learn to produce x_{t-3} - y{t} as the
    output of time step t.

    This method does not call Model.fit, but the dictionary it returns alongside
    the model will be passed as extra arguments whenever Model.fit is called.
    This can be used to, for example, set the batch size or use early stopping.

    :param input_shape: The shape of the inputs to the model.
    :param n_outputs: The number of outputs from the model.
    :return: A tuple of (neural network, Model.fit keyword arguments)
    """
    model = tensorflow.keras.models.S

In [ ]:
# %load test_nn.py
#!/usr/bin/env python

# In[1]:


import os
import numpy as np
import h5py
from tensorflow.keras.datasets import mnist
from tensorflow.keras.utils import to_categorical

# Load MNIST dataset
(train_input, train_output), (test_input, test_output) = mnist.load_data()

# Expand dimensions to make compatible with CNN input shape
train_input = np.expand_dims(train_input, axis=-1)
test_input = np.expand_dims(test_input, axis=-1)

# Convert labels to one-hot encoded vectors
train_output = to_categorical(train_output).astype(np.float32)
test_output = to_categorical(test_output).astype(np.float32)

# Create 'data' directory if it doesn't exist
if not os.path.exists('data'):  # For Assignment data is provided in hdf5 format
    os.makedirs('data')

# Save preprocessed data to HDF5 file
with h5py.File('data/mnist.hdf5', 'w') as f:
    # Include only every 100th training/testing example to limit dataset size
    train = f.create_group("train")
    train.create_dataset("input", compression="gzip", data=train_input[::100])
    train.create_dataset("output", compression="gzip", data=train_output[::100])
    test = f.create_group("test")
    test.create_dataset("input", compression="gzip", data=test_input[::100])
    test.create_dataset("output", compression="gzip", data=test_output[::100])


# In[2]:


import json
import re
import h5py
import numpy as np
import pandas as pd

# Download .csv files from
    #     https://archive.ics.uci.edu/ml/datasets/YouTube+Spam+Collection
names = ["1-Psy", "2-KatyPerry", "3-LMFAO","4-Eminem", "5-Shakira"]  # For Assignment all 5 datasets are provided
dfs = [pd.read_csv("data/Youtube0{0}.csv".format(name)) for name in names]
tokenize = re.compile(r"\d+|[^\d\W]+|\S").findall
dfs_tokenized = [[tokenize(comment) for comment in df["CONTENT"]]
                 for df in dfs]

index_to_token = [''] + sorted(set(token
                                   for comments in dfs_tokenized
                                   for tokens in comments
                                   for token in tokens))

token_to_index = {c: i for i, c in enumerate(index_to_token)}

max_tokens = max(len(tokens)
                 for comments in dfs_tokenized
                 for tokens in comments)

with h5py.File('data/youtube-comments.hdf5', 'w') as f:
    f.attrs["vocabulary"] = json.dumps(index_to_token)
    for name, df, comments in zip(names, dfs, dfs_tokenized):
        matrix_in = np.zeros(shape=(len(comments), max_tokens))
        for i, tokens in enumerate(comments):
            for j, token in enumerate(tokens):
                matrix_in[i, j] = token_to_index[token]
        matrix_out = df["CLASS"].values.reshape((-1, 1))
        group = f.create_group(name)
        group.create_dataset("input", compression="gzip", data=matrix_in)
        group.create_dataset("output", compression="gzip", data=matrix_out)


# In[3]:


import os
import json

import h5py
import numpy as np
import pytest
import tensorflow

import nn


@pytest.fixture(autouse=True)
def set_seeds():
    os.environ["TF_DETERMINISTIC_OPS"] = "1"
    tensorflow.random.set_seed(42)
    tensorflow.config.threading.set_intra_op_parallelism_threads(1)
    tensorflow.config.threading.set_inter_op_parallelism_threads(1)


def test_toy_rnn(capsys):
    n_train = 20
    n_test = 10
    n_timesteps = 20
    n_features = 2

    # create random input for train and test
    train_in = np.random.randint(1, 11, (n_train, n_timesteps, n_features))
    test_in = np.random.randint(1, 11, (n_test, n_timesteps, n_features))

    # deterministically create output from the random input
    def out(matrix_in):
        matrix_out = np.zeros(shape=matrix_in.shape[:-1] + (1,))
        for i, example in enumerate(matrix_in):
            for j, [_, x1] in enumerate(example):
                [x0, _] = example[j - 3] if j >= 3 else [0., 0.]
                matrix_out[i, j] = x0 - x1
        return matrix_out
    train_out = out(train_in)
    test_out = out(test_in)

    # request a model
    input_shape = train_in.shape[1:]
    (_, _, n_outputs) = train_out.shape
    model, kwargs = nn.create_toy_rnn(input_shape, n_outputs)

    # check that model contains a recurrent layer
    assert any(is_recurrent(layer) for layer in layers(model))

    # check that model contains no convolutional layers
    assert all(not is_convolution(layer) for layer in layers(model))

    # check that output type and loss are appropriate
    assert "mean" in loss_name(model)
    assert output_activation(model) == tensorflow.keras.activations.linear

    # set training data, epochs and validation data
    kwargs.update(x=train_in, y=train_out,
                  epochs=20, validation_data=(test_in, test_out))

    # call fit, including any arguments supplied alongside the model
    model.fit(**kwargs)

    # make sure error is low enough
    rmse = root_mean_squared_error(model.predict(test_in), test_out)
    with capsys.disabled():
        print("\n{:.1f} RMSE for RNN on toy problem".format(rmse))
    assert rmse < 2


def test_image_cnn(capsys):

    with h5py.File("data/mnist.hdf5", 'r') as f:
        train = f["train"]
        train_out = np.array(train["output"])
        train_in = np.array(train["input"])
        test = f["test"]
        test_out = np.array(test["output"])
        test_in = np.array(test["input"])

    # request a model
    input_shape = train_in.shape[1:]
    (_, n_outputs) = train_out.shape
    model, kwargs = nn.create_mnist_cnn(input_shape, n_outputs)

    # check that model contains a convolutional layer
    assert any(is_convolution(layer) for layer in layers(model))

    # check that model contains no recurrent layers
    assert all(not is_recurrent(layer) for layer in layers(model))

    # check that output type and loss are appropriate
    assert "categorical" in loss_name(model)
    assert output_activation(model) == tensorflow.keras.activations.softmax

    # set training data, epochs and validation data
    kwargs.update(x=train_in, y=train_out,
                  epochs=10, validation_data=(test_in, test_out))

    # call fit, including any arguments supplied alongside the model
    model.fit(**kwargs)

    # make sure accuracy is high enough
    accuracy = multi_class_accuracy(model.predict(test_in), test_out)
    with capsys.disabled():
        print("\n{:.1%} accuracy for CNN on MNIST sample".format(accuracy))
    assert accuracy > 0.8


def test_text_rnn(capsys):

    with h5py.File("data/youtube-comments.hdf5", 'r') as f:
        vocabulary = json.loads(f.attrs["vocabulary"])
        train = f["1-Psy"]
        train_in = np.array(train["input"])[:, :200]
        train_out = np.array(train["output"])
        test = f["5-Shakira"]
        test_in = np.array(test["input"])[:, :200]
        test_out = np.array(test["output"])

    # request a model
    model, kwargs = nn.create_youtube_comment_rnn(vocabulary=vocabulary,
                                                  n_outputs=1)

    # check that model contains a recurrent layer
    assert any(is_recurrent(layer) for layer in layers(model))

    # check that model contains no convolutional layers
    assert all(not is_convolution(layer) for layer in layers(model))

    # check that output type and loss are appropriate
    assert any(x in loss_name(model) for x in ["hinge", "crossentropy"])
    assert output_activation(model) == tensorflow.keras.activations.sigmoid

    # set training data, epochs and validation data
    kwargs.update(x=train_in, y=train_out,
                  epochs=10, validation_data=(test_in, test_out))

    # call fit, including any arguments supplied alongside the model
    model.fit(**kwargs)

    # make sure accuracy is high enough
    accuracy = binary_accuracy(model.predict(test_in), test_out)
    with capsys.disabled():
        print("\n{:.1%} accuracy for RNN on Youtube comments".format(accuracy))
    assert accuracy > 0.8


def test_text_cnn(capsys):
    # The data below was obtained as in test_text_rnn
    with h5py.File("data/youtube-comments.hdf5", 'r') as f:
        vocabulary = json.loads(f.attrs["vocabulary"])
        train = f["1-Psy"]
        train_in = np.array(train["input"])[:, :200]
        train_out = np.array(train["output"])
        test = f["5-Shakira"]
        test_in = np.array(test["input"])[:, :200]
        test_out = np.array(test["output"])

    # request a model
    model, kwargs = nn.create_youtube_comment_cnn(vocabulary=vocabulary,
                                                  n_outputs=1)

    # check that model contains a convolutional layer
    assert any(is_convolution(layer) for layer in layers(model))

    # check that model contains no recurrent layers
    assert all(not is_recurrent(layer) for layer in layers(model))

    # check that output type and loss are appropriate
    assert any(x in loss_name(model) for x in ["hinge", "crossentropy"])
    assert output_activation(model) == tensorflow.keras.activations.sigmoid

    # set training data, epochs and validation data
    kwargs.update(x=train_in, y=train_out,
                  epochs=10, validation_data=(test_in, test_out))

    # call fit, including any arguments supplied alongside the model
    model.fit(**kwargs)

    # make sure accuracy is high enough
    accuracy = binary_accuracy(model.predict(test_in), test_out)
    with capsys.disabled():
        print("\n{:.1%} accuracy for CNN on Youtube comments".format(accuracy))
    assert accuracy > 0.8


def layers(model: tensorflow.keras.models.Model):
    return [x.layer if isinstance(x, tensorflow.keras.layers.Wrapper) else x
            for x in model.layers]


def is_convolution(layer: tensorflow.keras.layers.Layer):
    return layer.__class__.__name__.startswith('Conv')


def is_recurrent(layer: tensorflow.keras.layers.Layer):
    return isinstance(layer, tensorflow.keras.layers.RNN)


def loss_name(model):
    if isinstance(model.loss, str):
        loss = getattr(tensorflow.keras.losses, model.loss)
    else:
        loss = model.loss
    return loss.__name__.lower()


def output_activation(model: tensorflow.keras.models.Model):
    return model.layers[-1].activation


def root_mean_squared_error(system: np.ndarray, human: np.ndarray):
    return ((system - human) ** 2).mean() ** 0.5


def multi_class_accuracy(system: np.ndarray, human: np.ndarray):
    return np.mean(np.argmax(system, axis=1) == np.argmax(human, axis=1))


def binary_accuracy(system: np.ndarray, human: np.ndarray):
    return np.mean(np.round(system) == human)



In [10]:
!pytest

============================= test session starts ==============================
platform darwin -- Python 3.10.9, pytest-7.1.2, pluggy-1.0.0
rootdir: /Users/slakshmanan/Playspace/uoams/neuralnetworks/convolution_and_recurrent_networks
plugins: anyio-3.5.0
collected 4 items                                                              

test_nn.py 
1.4 RMSE for RNN on toy problem
.
92.0% accuracy for CNN on MNIST sample
.
83.2% accuracy for RNN on Youtube comments
.
85.7% accuracy for CNN on Youtube comments
.                                                          [100%]

=============================== warnings summary ===============================
test_nn.py: 25 warnings
  /Users/slakshmanan/anaconda3/lib/python3.10/random.py:370: DeprecationWarning: non-integer arguments to randrange() have been deprecated since Python 3.10 and will be removed in a subsequent version
    return self.randrange(a, b+1)

-- Docs: https://docs.pytest.org/en/stable/how-to/capture-warnings.html
=======